# 🫀 心臟衰竭生存風險預測 API 教學：實作 `/predict` 重症死亡風險預測端點

> 對象：修習隨機森林分類器 / 模型部署課程的學生
> 對應程式碼：`app.py` 第 42~90 行的 `predict_endpoint`
>
> 這份 Notebook 會帶你**一步步拆解**臨床推論流程：從輸入 11 項生化生理指標防呆驗證、組裝特徵向量、呼叫 `model.predict_proba` 計算惡化死亡機率，到急診處置風險分級指引。

## 🎯 學習目標

- [ ] 理解臨床 11 項關鍵指標（如射血分數 EF、血清肌酸酐、CPK 激酶等）之生理意義與極限值防呆
- [ ] 看懂 Pydantic 的 `HeartPatientData` 欄位驗證設定 (`ge`, `le`)
- [ ] 掌握呼叫 `model.predict_proba()` 提取重症惡化死亡機率 ($P(Y=1|X)$)
- [ ] 建立臨床三級急診警示機制（🔴重症高死亡風險 / 🟡中度監控風險 / 🟢病情相對平穩）
- [ ] 能用 `TestClient` 呼叫真實 API 端點並驗證合法與異常輸入

## 📌 背景：`/predict` 完整預測流程

```
11項臨床指標 [age, anaemia, cpk, diabetes, ef, hbp, platelets, creatinine, sodium, sex, smoking]
       │
       ▼
1. Pydantic 數值範圍防呆檢驗 ──> 確保生化指標合理性 (如 EF 10~80%)
       │
       ▼
2. model.predict_proba(...)  ──> 取重症死亡機率 P (索引 1)
       │
       ▼
3. 臨床急診分級              ──> P >= 0.6: 🔴重症高風險 / P >= 0.35: 🟡中度 / 🟢平穩
       │
       ▼
回傳 RiskPredictionResponse 結構化 JSON
```

In [ ]:
# ============================================
# 0. 載入套件與模型
# ============================================
import os, joblib
import numpy as np
from pprint import pprint

MODEL_PATH = "heart_failure_model.joblib"
if not os.path.exists(MODEL_PATH):
    from train_save import train_and_save_model
    train_and_save_model(model_path=MODEL_PATH)

meta = joblib.load(MODEL_PATH)
model = meta["model"]
print("✅ 成功載入心臟衰竭隨機森林模型！")

---
## Part 1：手動測試單筆資料推論

模擬一位高風險重症病患：65歲、射血分數僅 20%、血清肌酸酐 2.5 mg/dL、無抽菸。

In [ ]:
# 11項指標順序：age, anaemia, cpk, diabetes, ef, hbp, platelets, creatinine, sodium, sex, smoking
sample_patient = np.array([[
    65.0,  # age
    1,     # anaemia (有貧血)
    250.0, # cpk
    1,     # diabetes (有糖尿病)
    20.0,  # ejection_fraction (收縮功能嚴重衰退)
    1,     # high_blood_pressure (有高血壓)
    210000.0, # platelets
    2.5,   # serum_creatinine (腎功能指標嚴重偏高)
    132.0, # serum_sodium
    1,     # sex (男性)
    0      # smoking (不抽菸)
]])

prob = float(model.predict_proba(sample_patient)[0, 1])
print(f"預估惡化死亡風險機率：{prob * 100:.2f}%")

---
## Part 2：定義 Pydantic 資料模型

使用 Pydantic 對 11 項醫療數值實施範圍防呆驗證。

In [ ]:
from pydantic import BaseModel, Field

class HeartPatientData(BaseModel):
    age: float = Field(..., ge=18, le=110, description="年齡 (歲)", example=65)
    anaemia: int = Field(..., ge=0, le=1, description="貧血 (0:無, 1:有)", example=0)
    cpk: float = Field(..., ge=10, le=10000, description="肌酸激酶 CPK (mcg/L)", example=160)
    diabetes: int = Field(..., ge=0, le=1, description="糖尿病史 (0:無, 1:有)", example=0)
    ejection_fraction: float = Field(..., ge=10, le=80, description="射血分數 EF (%)", example=35)
    high_blood_pressure: int = Field(..., ge=0, le=1, description="高血壓史 (0:無, 1:有)", example=1)
    platelets: float = Field(..., ge=20000, le=900000, description="血小板 (platelets/mL)", example=250000)
    serum_creatinine: float = Field(..., ge=0.4, le=10.0, description="血清肌酸酐 (mg/dL)", example=1.8)
    serum_sodium: float = Field(..., ge=110, le=150, description="血清鈉 (mEq/L)", example=136)
    sex: int = Field(..., ge=0, le=1, description="生理性別 (0:女性, 1:男性)", example=1)
    smoking: int = Field(..., ge=0, le=1, description="抽菸 (0:無, 1:有)", example=0)

class RiskPredictionResponse(BaseModel):
    mortality_risk_probability: float
    risk_category: str
    clinical_alert: str

print("✅ Pydantic 醫療資料模型宣告完成！")

---
## Part 3：建立 FastAPI App 與 `/predict` 預測端點

In [ ]:
from fastapi import FastAPI

app = FastAPI(title="Heart Failure Prediction API")

@app.post("/predict", response_model=RiskPredictionResponse)
def predict_endpoint(patient: HeartPatientData):
    raw_features = np.array([[
        patient.age, patient.anaemia, patient.cpk, patient.diabetes,
        patient.ejection_fraction, patient.high_blood_pressure, patient.platelets,
        patient.serum_creatinine, patient.serum_sodium, patient.sex, patient.smoking
    ]])
    
    prob = float(model.predict_proba(raw_features)[0, 1])
    
    if prob >= 0.6:
        category = "🔴 重症高死亡風險 (Critical High Risk)"
        alert = "肌酸酐或射血分數呈現心腎衰竭徵兆，建議即刻送加護病房 (ICU) 或安排急診心導管/強心利尿劑處置。"
    elif prob >= 0.35:
        category = "🟡 中度監控風險 (Moderate Risk)"
        alert = "心臟收縮功能受損，需密切追蹤電解質平衡與每日體重尿量變化。"
    else:
        category = "🟢 病情相對平穩 (Stable / Low Risk)"
        alert = "各項器官灌流指數尚可，持續維持常規慢性心衰竭藥物治療。"
        
    return RiskPredictionResponse(
        mortality_risk_probability=round(prob, 4),
        risk_category=category,
        clinical_alert=alert
    )

print("✅ /predict 端點建立完成！")

---
## Part 4：使用 `TestClient` 模擬呼叫 `/predict` 端點

In [ ]:
from fastapi.testclient import TestClient

client = TestClient(app)

test_payload = {
    "age": 60,
    "anaemia": 0,
    "cpk": 250,
    "diabetes": 0,
    "ejection_fraction": 38,
    "high_blood_pressure": 0,
    "platelets": 260000,
    "serum_creatinine": 1.1,
    "serum_sodium": 137,
    "sex": 1,
    "smoking": 0
}

response = client.post("/predict", json=test_payload)
print("狀態碼：", response.status_code)
print("回應結果：")
pprint(response.json())